In [ ]:
%pip install torch torch_geometric snappy pandas numpy

In [1]:
from data_utils import KnotDataset, KnotDataLoaders
from models.KnotGCN import KnotGCN
from models.KnotGAT import KnotGAT
from models.KnotGIN import KnotGIN
from training_utils import run_exp_pyg
from config import TASKS, TRAINING
import torch.nn as nn
import torch
import pandas as pd

/Users/cretuluca/uni/licenta/.venv/lib/python3.11/site-packages/plink/gui.py:33: UserWarning: Plink failed to import tkinter, GUI will not be available
  warnings.warn('Plink failed to import tkinter, GUI will not be available')
/Users/cretuluca/uni/licenta/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CSV_PATH = './datasets/knotinfo_aug.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODELS = {
    'GCN': KnotGCN,
    'GAT': KnotGAT,
    'GIN': KnotGIN,
}

results = {}

In [3]:
for task_name in TASKS:
    print(f"\n{'='*60}")
    print(f"Task: {task_name}")
    print(f"{'='*60}")

    dataset = KnotDataset(root='.', csv_path=CSV_PATH, task_name=task_name)
    loaders = KnotDataLoaders(
        dataset,
        batch_size=TRAINING['batch_size'],
        split=TRAINING['split'],
        seed=TRAINING['seed'],
    )

    results[task_name] = {}

    for model_name, ModelClass in MODELS.items():
        print(f"\n--- {model_name} ---")
        model = ModelClass(
            hidden_dim=TRAINING['hidden_dim'],
            num_classes=dataset.n_classes,
            num_layers=TRAINING['num_layers'],
        )

        stats = run_exp_pyg(
            model, loaders.train, loaders.val, loaders.test,
            loss_fct=nn.CrossEntropyLoss(),
            lr=TRAINING['lr'],
            num_epochs=TRAINING['num_epochs'],
            patience=TRAINING['patience'],
            DEVICE=DEVICE,
        )

        results[task_name][model_name] = {
            'test_acc': stats['test_acc'],
            'test_loss': stats['test_loss'],
            'best_val_acc': max(stats['val_acc']),
        }

Processing...



Task: Alternating


/Users/cretuluca/uni/licenta/invariant-prediction/data_utils.py:74: DtypeWarning: Columns (0: Conway Name, 1: Conway Notation, 2: Tetrahedral Census Name, 3: Crosscap Number, 4: Bridge Index, 5: Nakanishi Index, 6: Super Bridge Index, 7: A-Polynomial, 8: Genus-4D, 9: Genus-4D (Top.), 10: Crosscap Number-4D, 11: Concordance Genus, 12: Concordance Order (Alg.), 13: Concordance Order, 14: Concordance Order (Top.), 15: Max. Cusp Volume, 16: Longitude Translation, 17: Meridinal Translation, 18: Longitude Length, 19: Meridian Length, 20: Other Short Geodesics, 21: Chern-Simons Invariant, 22: Turaev Genus, 23: Monodromy, 24: Small or Large, 25: Positive Braid, 26: Positive, 27: SQ-Positive, 28: Q-Positive, 29: Pos. Braid Notation, 30: Pos. PD Notation, 31: SQ-Postive Braid, 32: Q-Positive Braid, 33: Clasp Number, 34: Nu, 35: Quasialternating, 36: Almost Alternating, 37: Montesinos Notation, 38: Boundary Slopes, 39: Pretzel Notation, 40: Unknotting Number (Alg.), 41: Mosaic/Tile-Number, 42: Al

[Alternating] Processed 75900 graphs, skipped 0


Done!



--- GCN ---

Model architecture:
KnotGCN(
  (embed): Embedding(2, 128)
  (convs): ModuleList(
    (0-3): 4 x GCNConv(128, 128)
  )
  (readout): Linear(in_features=128, out_features=2, bias=True)
)

Start training:
[Epoch 1] train loss: 0.448 acc: 0.836 | val loss: 0.433 acc: 0.845
[Epoch 2] train loss: 0.447 acc: 0.836 | val loss: 0.430 acc: 0.845


KeyboardInterrupt: 

In [ ]:
rows = []
for task_name, model_results in results.items():
    row = {'Task': task_name}
    for model_name, metrics in model_results.items():
        row[model_name] = f"{metrics['test_acc']:.3f}"
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('Task')
print(results_df.to_string())
results_df.to_csv('results.csv')